In [ ]:
#tfidf and matrix tiling

In [ ]:
import re
import time
import numpy as np
import cupy as cp
import pandas as pd



assert cp.cuda.runtime.getDeviceCount() > 0, "No CUDA GPU detected!"
with cp.cuda.Device(0) as dev:
    props = cp.cuda.runtime.getDeviceProperties(dev.id)
    print(f"[GPU] Using: {props['name'].decode() if isinstance(props['name'], bytes) else props['name']}")


def clean_text(s):
    if not isinstance(s, str):
        s = str(s)
    s = s.lower()
    s = re.sub(r"[^\w\s$]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_vocab(sentences):
    vocab = {}
    for s in sentences:
        for w in s.split():
            if w not in vocab:
                vocab[w] = len(vocab)
    return vocab

# TF–IDF on GPU
def compute_tfidf_gpu(sentences, vocab):
    n_docs = len(sentences)
    vocab_size = len(vocab)
    mat = cp.zeros((n_docs, vocab_size), dtype=cp.float32)

    for i, s in enumerate(sentences):
        words = s.split()
        for w in words:
            if w in vocab:
                mat[i, vocab[w]] += 1.0

    row_sums = cp.sum(mat, axis=1, keepdims=True) + 1e-9
    tf = mat / row_sums

    df = cp.sum(mat > 0, axis=0) + 1.0
    idf = cp.log((n_docs + 1.0) / df)

    tfidf = tf * idf
    # Normalize TF-IDF
    tfidf = tfidf / (cp.linalg.norm(tfidf, axis=1, keepdims=True) + 1e-9)
    return tfidf

# -------------------------
# NMF
def nmf_gpu(V, rank=50, iters=200, verbose=False):
    m, n = V.shape
    W = cp.random.rand(m, rank).astype(cp.float32) * 0.01
    H = cp.random.rand(rank, n).astype(cp.float32) * 0.01

    for i in range(iters):
        WH = W @ H
        H *= (W.T @ V) / (W.T @ WH + 1e-9)
        WH = W @ H
        W *= (V @ H.T) / (WH @ H.T + 1e-9)

        if verbose and ((i+1) % 100 == 0 or i == 0):
            loss = float(cp.linalg.norm(V - W @ H))
            print(f"[NMF] Iter {i+1}, Loss={loss:.4f}")
    return W, H
 
# CUDA Kernels

TILE = 16
matmul_tiled_code = '''
extern "C" __global__
void matmul_tiled(const float* A, const float* B, float* C, int M, int K, int N) {
    __shared__ float sA[16][16];
    __shared__ float sB[16][16];

    int row = blockIdx.y * 16 + threadIdx.y;
    int col = blockIdx.x * 16 + threadIdx.x;

    float value = 0.0f;

    for (int t = 0; t < (K + 16 - 1)/16; t++) {
        int tiled_row = row;
        int tiled_col = t*16 + threadIdx.x;
        if(tiled_row < M && tiled_col < K) sA[threadIdx.y][threadIdx.x] = A[tiled_row*K + tiled_col];
        else sA[threadIdx.y][threadIdx.x] = 0.0f;

        tiled_row = t*16 + threadIdx.y;
        tiled_col = col;
        if(tiled_row < K && tiled_col < N) sB[threadIdx.y][threadIdx.x] = B[tiled_row*N + tiled_col];
        else sB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for(int k=0;k<16;k++) value += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }

    if(row < M && col < N) C[row*N + col] = value;
}
'''
matmul_tiled_kernel = cp.RawKernel(matmul_tiled_code, 'matmul_tiled')

def matmul_manual(A, B):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, "Matrix dimensions mismatch for matmul"
    C = cp.zeros((M, N), dtype=cp.float32)
    block = (TILE, TILE, 1)
    grid = ((N+TILE-1)//TILE, (M+TILE-1)//TILE, 1)
    matmul_tiled_kernel(grid, block, (A, B, C, np.int32(M), np.int32(K), np.int32(N)))
    return C

# Softmax kernel
softmax_kernel_code = '''
extern "C" __global__
void row_softmax(float* X, int M, int N){
    int row = blockDim.x * blockIdx.x + threadIdx.x;
    if(row < M){
        float max_val = -1e20f;
        for(int j=0;j<N;++j){
            float v = X[row*N+j];
            max_val = v > max_val ? v : max_val;
        }
        float sum_exp=0.0f;
        for(int j=0;j<N;++j){
            float e = __expf(X[row*N+j]-max_val);
            X[row*N+j] = e;
            sum_exp += e;
        }
        float inv=1.0f/(sum_exp+1e-9f);
        for(int j=0;j<N;++j){
            X[row*N+j]*=inv;
        }
    }
}
'''
softmax_kernel = cp.RawKernel(softmax_kernel_code, 'row_softmax')
def softmax_manual(X):
    M, N = X.shape
    block = (128, 1, 1)
    grid = ((M+127)//128, 1, 1)
    softmax_kernel(grid, block, (X, np.int32(M), np.int32(N)))
    return X

# ReLU kernel
relu_kernel_code = '''
extern "C" __global__
void relu(float* X, int MN){
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if(idx < MN){
        float v = X[idx];
        X[idx] = v > 0.0f ? v : 0.0f;
    }
}
'''
relu_kernel = cp.RawKernel(relu_kernel_code, 'relu')
def relu_manual(X):
    MN = X.size
    block = (256, 1, 1)
    grid = ((MN+255)//256, 1, 1)
    relu_kernel(grid, block, (X, np.int32(MN)))
    return X

# Transformer Classifier

class TransformerClassifierGPU:
    def __init__(self, input_dim, n_classes, d_model=64):
        self.d_model = d_model
        self.input_dim = input_dim
        self.n_classes = n_classes
        self.W_embed = cp.random.randn(input_dim, d_model).astype(cp.float32) * 0.01
        self.W_Q = cp.random.randn(d_model, d_model).astype(cp.float32) * 0.01
        self.W_K = cp.random.randn(d_model, d_model).astype(cp.float32) * 0.01
        self.W_V = cp.random.randn(d_model, d_model).astype(cp.float32) * 0.01
        self.W_ff = cp.random.randn(d_model, d_model).astype(cp.float32) * 0.01
        self.b_ff = cp.zeros(d_model, dtype=cp.float32)
        self.W_out = cp.random.randn(d_model, n_classes).astype(cp.float32) * 0.01
        self.b_out = cp.zeros(n_classes, dtype=cp.float32)

    def forward(self, X):
        X_emb = matmul_manual(X, self.W_embed)
        Q = matmul_manual(X_emb, self.W_Q)
        K = matmul_manual(X_emb, self.W_K)
        V = matmul_manual(X_emb, self.W_V)

        scores = matmul_manual(Q, K.T) / cp.sqrt(cp.float32(self.d_model))
        attn = softmax_manual(scores.copy())
        attn_out = matmul_manual(attn, V)

        X_res = X_emb + attn_out
        mean = cp.mean(X_res, axis=1, keepdims=True)
        std = cp.std(X_res, axis=1, keepdims=True) + 1e-6
        X_norm = (X_res - mean) / std

        FF_pre = matmul_manual(X_norm, self.W_ff) + self.b_ff
        FF = relu_manual(FF_pre.copy())
        X_ff = X_norm + FF

        logits = matmul_manual(X_ff, self.W_out) + self.b_out

        cache = {
            'X': X, 'X_emb': X_emb, 'Q': Q, 'K': K, 'V': V,
            'scores': scores, 'attn': attn, 'attn_out': attn_out,
            'X_res': X_res, 'mean': mean, 'std': std, 'X_norm': X_norm,
            'FF_pre': FF_pre, 'FF': FF, 'X_ff': X_ff, 'logits': logits
        }
        return logits, X_ff, cache

    def backward(self, cache, y, logits, lr=0.001, weight_decay=1e-5):
        m = y.size
        grad_logits = cp.zeros_like(logits)
        grad_logits[cp.arange(m), y] = -1.0 / m
        exp_logits = cp.exp(logits - cp.max(logits, axis=1, keepdims=True))
        grad_logits += exp_logits / cp.sum(exp_logits, axis=1, keepdims=True)

        grad_norm = cp.sqrt(cp.sum(grad_logits ** 2))
        if grad_norm > 1.0:
            grad_logits *= 1.0 / grad_norm

        grad_W_out = matmul_manual(cache['X_ff'].T, grad_logits)
        grad_b_out = cp.sum(grad_logits, axis=0)

        grad_X_ff = matmul_manual(grad_logits, self.W_out.T)
        grad_FF = grad_X_ff * (cache['FF_pre'] > 0).astype(cp.float32)
        grad_W_ff = matmul_manual(cache['X_norm'].T, grad_FF)
        grad_b_ff = cp.sum(grad_FF, axis=0)

        grad_X_norm = grad_X_ff + grad_FF
        grad_X_res = grad_X_norm / cache['std']
        grad_mean = -cp.sum(grad_X_norm / cache['std'], axis=1, keepdims=True)
        grad_std = -cp.sum(grad_X_norm * (cache['X_res'] - cache['mean']) / (cache['std'] ** 2), axis=1, keepdims=True)
        grad_X_res += (grad_mean + grad_std * (cache['X_res'] - cache['mean']) / cache['std']) / cache['X_res'].shape[1]

        grad_attn_out = grad_X_res
        grad_attn = matmul_manual(grad_attn_out, cache['V'].T)
        grad_V = matmul_manual(cache['attn'].T, grad_attn_out)

        grad_scores = grad_attn * cache['attn'] - cp.sum(grad_attn * cache['attn'], axis=1, keepdims=True)

        grad_Q = matmul_manual(grad_scores, cache['K']) / cp.sqrt(cp.float32(self.d_model))
        grad_K = matmul_manual(grad_scores.T, cache['Q']) / cp.sqrt(cp.float32(self.d_model))
        grad_V = matmul_manual(cache['attn'].T, grad_attn_out)
        grad_X_emb = matmul_manual(grad_Q, self.W_Q.T) + matmul_manual(grad_K, self.W_K.T) + matmul_manual(grad_V, self.W_V.T) + grad_X_res

        grad_W_embed = matmul_manual(cache['X'].T, grad_X_emb)
        grad_W_Q = matmul_manual(cache['X_emb'].T, grad_Q)
        grad_W_K = matmul_manual(cache['X_emb'].T, grad_K)
        grad_W_V = matmul_manual(cache['X_emb'].T, grad_V)

        self.W_out -= lr * (grad_W_out + weight_decay * self.W_out)
        self.b_out -= lr * grad_b_out
        self.W_ff -= lr * (grad_W_ff + weight_decay * self.W_ff)
        self.b_ff -= lr * grad_b_ff
        self.W_Q -= lr * (grad_W_Q + weight_decay * self.W_Q)
        self.W_K -= lr * (grad_W_K + weight_decay * self.W_K)
        self.W_V -= lr * (grad_W_V + weight_decay * self.W_V)
        self.W_embed -= lr * (grad_W_embed + weight_decay * self.W_embed)

        return {
            'W_out': grad_W_out, 'b_out': grad_b_out,
            'W_ff': grad_W_ff, 'b_ff': grad_b_ff,
            'W_Q': grad_W_Q, 'W_K': grad_W_K, 'W_V': grad_W_V,
            'W_embed': grad_W_embed
        }

def cross_entropy_loss(logits, y):
    maxl = cp.max(logits, axis=1, keepdims=True)
    exp_logits = cp.exp(logits - maxl)
    log_probs = logits - maxl - cp.log(cp.sum(exp_logits, axis=1, keepdims=True))
    return -cp.mean(log_probs[cp.arange(y.size), y])

def manual_split(X, y, val_ratio=0.15, seed=42):
    np.random.seed(seed)
    idx = np.arange(len(y))
    np.random.shuffle(idx)
    split = int(len(y) * (1 - val_ratio))
    train_idx, val_idx = idx[:split], idx[split:]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

if __name__ == "__main__":
    cp.random.seed(42)
    np.random.seed(42)

    # Load CSV
    try:
        df = pd.read_csv(r"C:\Users\Pavani Akshaya\OneDrive\Desktop\fin_data_1.csv")
    except FileNotFoundError:
        print("[ERROR] CSV file not found. Please check the file path.")
        exit(1)

    sentences = [clean_text(s) for s in df["Sentence"].astype(str).tolist()]

    def map_label(x):
        if pd.isna(x):
            return 1
        s = str(x).strip().lower()
        if s in ("positive", "pos", "p", "2", "+", "positive "):
            return 2
        if s in ("negative", "neg", "n", "0", "-", "negative "):
            return 0
        if s in ("neutral", "neu", "1", "neutral "):
            return 1
        if s.isdigit():
            v = int(s)
            if v <= 0: return 0
            if v == 1: return 1
            return 2
        return 1

    labels = df["Sentiment"].apply(map_label).astype(int).to_numpy()

    vocab = build_vocab(sentences)
    print(f"[DATA] {len(sentences)} docs, {len(vocab)} vocab, classes: {set(labels)}")

    tfidf = compute_tfidf_gpu(sentences, vocab)

    W, H = nmf_gpu(tfidf, rank=64, iters=200, verbose=True)
    X_gpu = W / (cp.linalg.norm(W, axis=1, keepdims=True) + 1e-9)

    n_classes = 3
    model = TransformerClassifierGPU(X_gpu.shape[1], n_classes)

    y_gpu = cp.asarray(labels)

    # Manual split
    X_np = cp.asnumpy(X_gpu)
    y_np = cp.asnumpy(y_gpu)
    X_tr, X_val, y_tr, y_val = manual_split(X_np, y_np, val_ratio=0.15)
    X_tr = cp.asarray(X_tr)
    X_val = cp.asarray(X_val)
    y_tr = cp.asarray(y_tr).astype(cp.int32)
    y_val = cp.asarray(y_val).astype(cp.int32)

    print(f"[SPLIT] Train={X_tr.shape[0]}, Val={X_val.shape[0]}")

    best_val_acc = 0.0
    best_model_state = None
    best_epoch = -1

    num_epochs = 200
    lr = 0.001

    for epoch in range(num_epochs):
        logits, X_ff, cache = model.forward(X_tr)
        loss = cross_entropy_loss(logits, y_tr)

        grads = model.backward(cache, y_tr, logits, lr=lr, weight_decay=1e-5)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            preds = cp.argmax(logits, axis=1)
            acc = float(cp.mean((preds == y_tr).astype(cp.float32))) * 100.0

            logits_val, _, _ = model.forward(X_val)
            preds_val = cp.argmax(logits_val, axis=1)
            val_acc = float(cp.mean((preds_val == y_val).astype(cp.float32))) * 100.0

            grad_norm = cp.sqrt(sum(cp.sum(g ** 2) for g in grads.values()))
            print(f"[TRANS] Epoch {epoch+1:03d} Loss={float(loss):.4f} TrainAcc={acc:.2f}% ValAcc={val_acc:.2f}% GradNorm={float(grad_norm):.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch + 1
                best_model_state = {
                    'W_embed': model.W_embed.copy(), 'W_Q': model.W_Q.copy(), 'W_K': model.W_K.copy(),
                    'W_V': model.W_V.copy(), 'W_ff': model.W_ff.copy(), 'b_ff': model.b_ff.copy(),
                    'W_out': model.W_out.copy(), 'b_out': model.b_out.copy()
                }
        if (epoch + 1) % 50 == 0:
            lr *= 0.5
    if best_epoch > 0:
        print(f"[TRANS] Best Val Acc: {best_val_acc:.2f}% at epoch {best_epoch}")
    else:
        print("[TRANS] No improvement recorded on validation set during training.")
    if best_model_state is not None:
        model.W_embed = best_model_state['W_embed']
        model.W_Q = best_model_state['W_Q']
        model.W_K = best_model_state['W_K']
        model.W_V = best_model_state['W_V']
        model.W_ff = best_model_state['W_ff']
        model.b_ff = best_model_state['b_ff']
        model.W_out = best_model_state['W_out']
        model.b_out = best_model_state['b_out']

    logits_final, _, _ = model.forward(X_gpu)
    preds_final = cp.argmax(logits_final, axis=1)
    overall_acc = float(cp.mean((preds_final == y_gpu).astype(cp.float32))) * 100.0
    print(f" Final overall accuracy (all data): {overall_acc:.2f}%")

[GPU] Using: NVIDIA GeForce RTX 3060 Laptop GPU
[DATA] 5842 docs, 11554 vocab, classes: {np.int64(0), np.int64(1), np.int64(2)}
[NMF] Iter 1, Loss=75.6329
[NMF] Iter 100, Loss=70.4380
[NMF] Iter 200, Loss=70.4177
[SPLIT] Train=4965, Val=877
[TRANS] Epoch 001 Loss=1.1062 TrainAcc=32.39% ValAcc=32.61% GradNorm=222.4986
[TRANS] Epoch 005 Loss=1.1048 TrainAcc=30.41% ValAcc=31.36% GradNorm=84.4740
[TRANS] Epoch 010 Loss=1.1058 TrainAcc=28.46% ValAcc=29.76% GradNorm=81.8529
[TRANS] Epoch 015 Loss=1.1053 TrainAcc=28.96% ValAcc=30.90% GradNorm=84.7285
[TRANS] Epoch 020 Loss=1.1014 TrainAcc=31.10% ValAcc=33.41% GradNorm=87.9645
[TRANS] Epoch 025 Loss=1.0970 TrainAcc=32.71% ValAcc=33.75% GradNorm=94.1694
[TRANS] Epoch 030 Loss=1.0945 TrainAcc=34.00% ValAcc=34.66% GradNorm=101.3235
[TRANS] Epoch 035 Loss=1.0931 TrainAcc=37.89% ValAcc=36.83% GradNorm=112.5003
[TRANS] Epoch 040 Loss=1.0981 TrainAcc=32.45% ValAcc=33.41% GradNorm=124.8149
[TRANS] Epoch 045 Loss=1.1110 TrainAcc=29.08% ValAcc=30.22% Gr